In [8]:
import os
from pathlib import Path
import glob
import math
import random
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.integrate import trapezoid
from scipy.signal import welch
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings("ignore")

In [9]:
# -------------------------------------------------------------
# ۰. Reproducibility (very important)
# -------------------------------------------------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(42)

In [10]:
# -------------------------------------------------------------
# ۱. Settings
# -------------------------------------------------------------
DATA_DIR = Path(r'F:\Github\pxsa\Digital-Biomarkers\Datasets\Multi-channel Wireless EEG Recordings of Young Adu')
OUTPUT_DIR = "./powerbi_exports"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FS = 128
EPOCH_SEC = 4
OVERLAP_SEC = 2
CHANNELS = ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']
BANDS = {
    'Delta': (0.5, 4),
    'Theta': (4, 8),
    'Alpha': (8, 13),
    'Beta': (13, 30),
    'Gamma': (30, 45)
}

In [11]:
# -------------------------------------------------------------
# ۲. Feature extraction
# -------------------------------------------------------------
def compute_bandpowers(epoch_data, fs):
    feats = {}
    n_samples, n_ch = epoch_data.shape
    for ch_idx, ch_name in enumerate(CHANNELS):
        freqs, psd = welch(epoch_data[:, ch_idx], fs=fs, nperseg=min(n_samples, fs*2))
        total_power = trapezoid(psd, freqs) + 1e-10

        for band_name, (f_low, f_high) in BANDS.items():
            idx_band = np.logical_and(freqs >= f_low, freqs <= f_high)
            band_pow = trapezoid(psd[idx_band], freqs[idx_band])
            feats[f"{ch_name}_{band_name}"] = band_pow / total_power
    return feats

In [12]:
# -------------------------------------------------------------
# ۳. Load data
# -------------------------------------------------------------
print("Loading data and extracting features...")
mat_files = glob.glob(os.path.join(DATA_DIR, "*.mat"))
records = []
subject_metadata = []

for file_path in mat_files:
    fname = os.path.basename(file_path).replace('.mat', '')
    if fname.startswith("ASub"):
        label, group = 1, "Anxiety"
    elif fname.startswith("ACSub"):
        label, group = 0, "Control"
    else:
        continue

    mat_content = loadmat(file_path)
    raw_data = None
    for k in mat_content:
        if not k.startswith("__") and isinstance(mat_content[k], np.ndarray):
            if mat_content[k].shape in [(38400, 14), (14, 38400)]:
                raw_data = mat_content[k]
                break
    if raw_data is None:
        continue
    if raw_data.shape[0] == 14:
        raw_data = raw_data.T

    step_samples = int((EPOCH_SEC - OVERLAP_SEC) * FS)
    win_samples = int(EPOCH_SEC * FS)
    n_epochs = (len(raw_data) - win_samples) // step_samples + 1

    for ep in range(n_epochs):
        start = ep * step_samples
        end = start + win_samples
        window = raw_data[start:end, :]
        ep_feats = compute_bandpowers(window, FS)
        ep_feats.update({
            'Subject_ID': fname,
            'Epoch_ID': ep,
            'Label': label,
            'Group': group
        })
        records.append(ep_feats)

    subject_metadata.append({
        'Subject_ID': fname,
        'Group': group,
        'Label': label,
        'Total_Epochs': n_epochs
    })

df_all = pd.DataFrame(records)
df_meta = pd.DataFrame(subject_metadata)
print(f"Total epochs: {len(df_all)}")

feat_cols = [c for c in df_all.columns if c not in ['Subject_ID', 'Epoch_ID', 'Label', 'Group']]
X = df_all[feat_cols].values
y = df_all['Label'].values
groups = df_all['Subject_ID'].values

Loading data and extracting features...
Total epochs: 5662


In [ ]:
# -------------------------------------------------------------
# ۴. Classical models
# -------------------------------------------------------------
models = {
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                             use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=150, max_depth=5, random_state=42),
    "SVM_RBF": SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=42)
}

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
all_predictions = {}

print("\nEvaluating classical models (subject-aware CV)...")
for name, model in models.items():
    print(f"  → {name}")
    y_pred = cross_val_predict(model, X, y, cv=sgkf, groups=groups, n_jobs=-1)
    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y, y_pred) * 100, 2),
        "Precision": round(precision_score(y, y_pred) * 100, 2),
        "Recall": round(recall_score(y, y_pred) * 100, 2),
        "F1_Score": round(f1_score(y, y_pred) * 100, 2)
    })
    all_predictions[name] = y_pred


Evaluating classical models (subject-aware CV)...
  → RandomForest
  → XGBoost
  → GradientBoosting
  → SVM_RBF
  → LogisticRegression
  → MLP


In [37]:
# -------------------------------------------------------------
# ۵. Stabilized Attention Model
# -------------------------------------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class AttentionClassifier(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=8, num_layers=2, dropout=0.25):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout * 0.5)
        )
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_encoder = PositionalEncoding(d_model, max_len=400, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*3,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.attn_pool = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.Tanh(),
            nn.Linear(d_model // 2, 1, bias=False)
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 2)
        )

    def forward(self, x, mask):
        B, T, _ = x.shape
        x = self.input_proj(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        cls_mask = torch.ones(B, 1, device=mask.device, dtype=mask.dtype)
        mask = torch.cat([cls_mask, mask], dim=1)
        x = self.pos_encoder(x)
        key_padding_mask = (mask == 0)
        x = self.transformer(x, src_key_padding_mask=key_padding_mask)
        attn_scores = self.attn_pool(x).squeeze(-1).masked_fill(key_padding_mask, -1e9)
        attn_weights = torch.softmax(attn_scores, dim=1)
        x = torch.sum(x * attn_weights.unsqueeze(-1), dim=1)
        return self.classifier(x)


class EpochSequenceDataset(Dataset):
    def __init__(self, df, feat_cols, subject_ids, max_len):
        self.sequences, self.labels, self.subjects = [], [], []
        for sid in subject_ids:
            sub = df[df['Subject_ID'] == sid].sort_values('Epoch_ID')
            self.sequences.append(sub[feat_cols].values.astype(np.float32))
            self.labels.append(sub['Label'].iloc[0])
            self.subjects.append(sid)
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        if len(seq) > self.max_len:
            seq = seq[:self.max_len]
        else:
            pad = np.zeros((self.max_len - len(seq), seq.shape[1]), dtype=np.float32)
            seq = np.vstack([seq, pad])
        mask = (seq.sum(axis=1) != 0).astype(np.float32)
        return torch.tensor(seq), torch.tensor(mask), torch.tensor(self.labels[idx], dtype=torch.long)


def train_one_seed(train_df, val_df, feat_cols, seed, epochs=40, batch_size=8, patience=7):
    seed_everything(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    max_len = max(train_df.groupby('Subject_ID').size().max(),
                  val_df.groupby('Subject_ID').size().max())

    train_subjects = train_df['Subject_ID'].unique()
    val_subjects = val_df['Subject_ID'].unique()

    train_ds = EpochSequenceDataset(train_df, feat_cols, train_subjects, max_len)
    val_ds   = EpochSequenceDataset(val_df,   feat_cols, val_subjects,   max_len)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size)

    model = AttentionClassifier(input_dim=len(feat_cols)).to(device)

    # Class weights
    labels = train_df.groupby('Subject_ID')['Label'].first().values
    counts = np.bincount(labels)
    weights = 1.0 / counts
    weights = weights / weights.sum() * len(counts)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32).to(device))

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    best_val_acc = 0.0
    best_state = None
    no_improve = 0

    for ep in range(epochs):
        model.train()
        for x, mask, yb in train_loader:
            x, mask, yb = x.to(device), mask.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x, mask), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # Validation
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, mask, yb in val_loader:
                x, mask, yb = x.to(device), mask.to(device), yb.to(device)
                preds = model(x, mask).argmax(1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        val_acc = correct / total if total > 0 else 0
        scheduler.step(val_acc)

        if val_acc > best_val_acc + 1e-4:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)
    return model, max_len, best_val_acc


print("\nEvaluating Stabilized Attention model (3 seeds)...")
unique_subjects = df_all['Subject_ID'].unique()
subject_labels = df_all.groupby('Subject_ID')['Label'].first().reindex(unique_subjects).values

# We will average probabilities from 3 seeds
attn_probs = np.zeros((len(df_all), 2))
seeds = [42]

for seed in seeds:
    print(f"  → Seed {seed}")
    fold_probs = np.zeros((len(df_all), 2))

    for train_idx, val_idx in StratifiedGroupKFold(5, shuffle=True, random_state=42).split(
            unique_subjects, subject_labels, groups=unique_subjects):

        train_subj = unique_subjects[train_idx]
        val_subj   = unique_subjects[val_idx]

        train_df = df_all[df_all['Subject_ID'].isin(train_subj)].copy()
        val_df   = df_all[df_all['Subject_ID'].isin(val_subj)].copy()

        scaler = StandardScaler()
        train_df[feat_cols] = scaler.fit_transform(train_df[feat_cols])
        val_df[feat_cols]   = scaler.transform(val_df[feat_cols])

        model, max_len, _ = train_one_seed(train_df, val_df, feat_cols, seed=seed)

        # Predict
        device = next(model.parameters()).device
        model.eval()
        val_ds = EpochSequenceDataset(val_df, feat_cols, val_subj, max_len)
        val_loader = DataLoader(val_ds, batch_size=8)

        with torch.no_grad():
            idx = 0
            for x, mask, _ in val_loader:
                x, mask = x.to(device), mask.to(device)
                probs = torch.softmax(model(x, mask), dim=1).cpu().numpy()
                for p in probs:
                    sid = val_subj[idx]
                    fold_probs[df_all['Subject_ID'] == sid] = p
                    idx += 1

    attn_probs += fold_probs

attn_probs /= len(seeds)
attn_preds = attn_probs.argmax(axis=1)

# Metrics
acc  = accuracy_score(y, attn_preds)
prec = precision_score(y, attn_preds)
rec  = recall_score(y, attn_preds)
f1   = f1_score(y, attn_preds)

results.append({
    "Model": "AttentionTransformer_Stabilized",
    "Accuracy": round(acc * 100, 2),
    "Precision": round(prec * 100, 2),
    "Recall": round(rec * 100, 2),
    "F1_Score": round(f1 * 100, 2)
})
all_predictions["AttentionTransformer_Stabilized"] = attn_preds


Evaluating Stabilized Attention model (3 seeds)...
  → Seed 42


In [38]:
# -------------------------------------------------------------
# ۶. Results & Power BI Exports
# -------------------------------------------------------------
df_results = pd.DataFrame(results).sort_values("F1_Score", ascending=False)
print("\n=== Final Model Comparison ===")
print(df_results.to_string(index=False))

best_model_name = df_results.iloc[0]["Model"]
print(f"\nBest model: {best_model_name}")

df_all['Prediction'] = all_predictions[best_model_name]

# 1. Subject Summary
subject_perf = df_all.groupby('Subject_ID').apply(
    lambda g: pd.Series({
        'Accuracy': accuracy_score(g['Label'], g['Prediction']),
        'Correct_Epochs': (g['Label'] == g['Prediction']).sum(),
        'Total_Epochs': len(g),
        'Predicted_Label': g['Prediction'].mode()[0],
        'True_Label': g['Label'].iloc[0]
    })
).reset_index()
df_subject_summary = pd.merge(df_meta, subject_perf, on='Subject_ID')
df_subject_summary.to_csv(os.path.join(OUTPUT_DIR, "Subject_Summary.csv"), index=False)

# 2. Overall Metrics
pd.DataFrame([
    {"Metric": "Accuracy",  "Value": round(accuracy_score(y, df_all['Prediction']) * 100, 2)},
    {"Metric": "Precision", "Value": round(precision_score(y, df_all['Prediction']) * 100, 2)},
    {"Metric": "Recall",    "Value": round(recall_score(y, df_all['Prediction']) * 100, 2)},
    {"Metric": "F1_Score",  "Value": round(f1_score(y, df_all['Prediction']) * 100, 2)},
]).to_csv(os.path.join(OUTPUT_DIR, "Overall_Metrics.csv"), index=False)

# 3. Confusion Matrix
cm = confusion_matrix(y, df_all['Prediction'])
pd.DataFrame([
    {'Actual': 'Control', 'Predicted': 'Control', 'Count': int(cm[0, 0])},
    {'Actual': 'Control', 'Predicted': 'Anxiety', 'Count': int(cm[0, 1])},
    {'Actual': 'Anxiety', 'Predicted': 'Control', 'Count': int(cm[1, 0])},
    {'Actual': 'Anxiety', 'Predicted': 'Anxiety', 'Count': int(cm[1, 1])}
]).to_csv(os.path.join(OUTPUT_DIR, "Confusion_Matrix.csv"), index=False)

# 4. Model Comparison
df_results.to_csv(os.path.join(OUTPUT_DIR, "Model_Comparison.csv"), index=False)

# 5. Feature Importance (placeholder)
pd.DataFrame({
    'Feature': feat_cols,
    'Channel': [f.split('_')[0] for f in feat_cols],
    'Band': [f.split('_')[1] for f in feat_cols],
    'Importance': 0.0
}).to_csv(os.path.join(OUTPUT_DIR, "Feature_Importance.csv"), index=False)

# 6. Bandpower long
bandpower_long = df_all.melt(
    id_vars=['Subject_ID', 'Group', 'Label', 'Prediction'],
    value_vars=feat_cols,
    var_name='Feature',
    value_name='Relative_Power'
)
bandpower_long['Channel'] = bandpower_long['Feature'].str.split('_').str[0]
bandpower_long['Band'] = bandpower_long['Feature'].str.split('_').str[1]
bandpower_long.drop(columns=['Feature']).to_csv(
    os.path.join(OUTPUT_DIR, "Channel_Bandpower_Long.csv"), index=False
)

# 7. Full predictions
df_all[['Subject_ID', 'Epoch_ID', 'Group', 'Label', 'Prediction']].to_csv(
    os.path.join(OUTPUT_DIR, "Epoch_Level_Predictions.csv"), index=False
)

print(f"\nAll files saved to: {OUTPUT_DIR}")
print("Done.")


=== Final Model Comparison ===
                          Model  Accuracy  Precision  Recall  F1_Score
AttentionTransformer_Stabilized     73.68      76.00   82.61     79.17
AttentionTransformer_Stabilized     73.68      76.00   82.61     79.17
AttentionTransformer_Stabilized     71.05      75.00   78.26     76.60
AttentionTransformer_Stabilized     65.79      75.00   65.22     69.77
AttentionTransformer_Stabilized     65.79      75.00   65.22     69.77
                        SVM_RBF     61.94      67.80   70.70     69.22
                   RandomForest     58.14      62.82   75.58     68.61
                            MLP     60.86      66.81   70.24     68.48
               GradientBoosting     59.22      64.72   71.72     68.04
                        XGBoost     59.04      64.68   71.23     67.80
             LogisticRegression     55.99      63.78   63.15     63.46
      Attention_Simple_HighPerf     60.53      72.22   56.52     63.41
           AttentionTransformer     55.26    

# Maybe

In [24]:
# -----------------------------
# 4.2 Attention-based model (PyTorch)
# -----------------------------
print("\nEvaluating Attention model...")

class EpochSequenceDataset(Dataset):
    def __init__(self, df, feat_cols, subject_ids, max_len=None):
        self.sequences = []
        self.labels = []
        self.subjects = []
        
        for sid in subject_ids:
            sub = df[df['Subject_ID'] == sid].sort_values('Epoch_ID')
            seq = sub[feat_cols].values.astype(np.float32)
            label = sub['Label'].iloc[0]
            self.sequences.append(seq)
            self.labels.append(label)
            self.subjects.append(sid)
        
        if max_len is None:
            self.max_len = max(len(s) for s in self.sequences)
        else:
            self.max_len = max_len
            
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        # pad / truncate
        if len(seq) > self.max_len:
            seq = seq[:self.max_len]
        else:
            pad = np.zeros((self.max_len - len(seq), seq.shape[1]), dtype=np.float32)
            seq = np.vstack([seq, pad])
        
        mask = (seq.sum(axis=1) != 0).astype(np.float32)  # 1 = real, 0 = pad
        return torch.tensor(seq), torch.tensor(mask), torch.tensor(self.labels[idx], dtype=torch.long)


class AttentionClassifier(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.attn_pool = nn.Linear(d_model, 1)          # simple attention pooling
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 2)
        )
        
    def forward(self, x, mask):
        # x: (B, T, F)
        x = self.input_proj(x)
        # create key_padding_mask (True = ignore)
        key_padding_mask = (mask == 0)
        x = self.transformer(x, src_key_padding_mask=key_padding_mask)
        
        # Attention pooling
        attn_weights = torch.softmax(self.attn_pool(x).squeeze(-1).masked_fill(key_padding_mask, -1e9), dim=1)
        x = torch.sum(x * attn_weights.unsqueeze(-1), dim=1)   # (B, d_model)
        
        return self.classifier(x)


def train_attention_model(train_df, val_df, feat_cols, epochs=40, batch_size=8, lr=1e-3):
    train_subjects = train_df['Subject_ID'].unique()
    val_subjects   = val_df['Subject_ID'].unique()
    
    max_len = max(
        train_df.groupby('Subject_ID').size().max(),
        val_df.groupby('Subject_ID').size().max()
    )
    
    train_ds = EpochSequenceDataset(train_df, feat_cols, train_subjects, max_len)
    val_ds   = EpochSequenceDataset(val_df,   feat_cols, val_subjects,   max_len)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = AttentionClassifier(input_dim=len(feat_cols)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    best_val_acc = 0
    best_state = None
    
    for ep in range(epochs):
        model.train()
        for x, mask, yb in train_loader:
            x, mask, yb = x.to(device), mask.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(x, mask)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
        
        # validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, mask, yb in val_loader:
                x, mask, yb = x.to(device), mask.to(device), yb.to(device)
                logits = model(x, mask)
                preds = logits.argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        val_acc = correct / total
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = model.state_dict()
    
    model.load_state_dict(best_state)
    return model, max_len


Evaluating Attention model...


In [25]:
# Subject-level 5-fold for Attention
unique_subjects = df_all['Subject_ID'].unique()
subject_labels  = df_all.groupby('Subject_ID')['Label'].first().reindex(unique_subjects).values

attn_preds = np.zeros(len(df_all), dtype=int)
attn_sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(attn_sgkf.split(unique_subjects, subject_labels, groups=unique_subjects)):
    train_subjects = unique_subjects[train_idx]
    val_subjects   = unique_subjects[val_idx]
    
    train_df = df_all[df_all['Subject_ID'].isin(train_subjects)]
    val_df   = df_all[df_all['Subject_ID'].isin(val_subjects)]
    
    # Scale features (important for NN)
    scaler = StandardScaler()
    train_df = train_df.copy()
    val_df   = val_df.copy()
    train_df[feat_cols] = scaler.fit_transform(train_df[feat_cols])
    val_df[feat_cols]   = scaler.transform(val_df[feat_cols])
    
    model, max_len = train_attention_model(train_df, val_df, feat_cols, epochs=35)
    
    # Predict on validation subjects
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    val_ds = EpochSequenceDataset(val_df, feat_cols, val_subjects, max_len)
    val_loader = DataLoader(val_ds, batch_size=8)
    
    with torch.no_grad():
        for i, (x, mask, _) in enumerate(val_loader):
            x, mask = x.to(device), mask.to(device)
            logits = model(x, mask)
            preds = logits.argmax(dim=1).cpu().numpy()
            
            # map back to original rows
            start = i * 8
            for j, pred in enumerate(preds):
                sid = val_subjects[start + j]
                mask_rows = df_all['Subject_ID'] == sid
                attn_preds[mask_rows] = pred

# Metrics for Attention
acc  = accuracy_score(y, attn_preds)
prec = precision_score(y, attn_preds)
rec  = recall_score(y, attn_preds)
f1   = f1_score(y, attn_preds)

results.append({
    "Model": "AttentionTransformer",
    "Accuracy": round(acc * 100, 2),
    "Precision": round(prec * 100, 2),
    "Recall": round(rec * 100, 2),
    "F1_Score": round(f1 * 100, 2)
})
all_predictions["AttentionTransformer"] = attn_preds

In [26]:
results

[{'Model': 'RandomForest',
  'Accuracy': 58.14,
  'Precision': 62.82,
  'Recall': 75.58,
  'F1_Score': 68.61},
 {'Model': 'XGBoost',
  'Accuracy': 59.04,
  'Precision': 64.68,
  'Recall': 71.23,
  'F1_Score': 67.8},
 {'Model': 'GradientBoosting',
  'Accuracy': 59.22,
  'Precision': 64.72,
  'Recall': 71.72,
  'F1_Score': 68.04},
 {'Model': 'SVM_RBF',
  'Accuracy': 61.94,
  'Precision': 67.8,
  'Recall': 70.7,
  'F1_Score': 69.22},
 {'Model': 'LogisticRegression',
  'Accuracy': 55.99,
  'Precision': 63.78,
  'Recall': 63.15,
  'F1_Score': 63.46},
 {'Model': 'MLP',
  'Accuracy': 60.86,
  'Precision': 66.81,
  'Recall': 70.24,
  'F1_Score': 68.48},
 {'Model': 'Attention_Simple_HighPerf',
  'Accuracy': 60.53,
  'Precision': 72.22,
  'Recall': 56.52,
  'F1_Score': 63.41},
 {'Model': 'AttentionTransformer',
  'Accuracy': 55.26,
  'Precision': 65.0,
  'Recall': 56.52,
  'F1_Score': 60.47},
 {'Model': 'AttentionTransformer',
  'Accuracy': 55.26,
  'Precision': 65.0,
  'Recall': 56.52,
  'F1_Sc